# SkyLedger Airlines — SQL Business Analysis (Gold Layer)

Twenty business-focused SQL queries built on the governed Gold datasets in `skyledger_airlines.gold` — `fact_bookings` (booking + flight-segment fact at the booking_id × line_number grain) joined to `dim_passenger`, `dim_route`, and `dim_airline`. Every query is titled, tested, and followed by a business interpretation and a suggested management action.

**Metric definitions (applied consistently):**

* Net revenue (before tax) = `ticket_price × (1 − discount)`
* Bookings = `COUNT(DISTINCT booking_id)`
* Active passengers = distinct passengers with at least one booking in the selected period
* Average booking value = net revenue ÷ distinct bookings

**Geography convention:** *passenger geography* = the passenger's home nation/region; *airline geography* = the carrier's home nation/region. Q20 compares the two explicitly.

*TPC-A is synthetic benchmark data created for training purposes — treat all findings as learning exercises, not real airline performance.*

## Q01 — Overall KPI Summary

Net revenue, distinct bookings, active passengers and average booking value across the full 1994–1998 booking history.

In [0]:
%sql
-- Q01 · Overall KPI summary
-- Net revenue, distinct bookings, active passengers, average booking value (all years)
SELECT
  ROUND(SUM(net_revenue), 2)                              AS net_revenue,
  COUNT(DISTINCT booking_id)                              AS total_bookings,
  COUNT(DISTINCT passenger_key)                          AS active_passengers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2) AS average_booking_value
FROM skyledger_airlines.gold.fact_bookings

**Interpretation.** Across 1994–1998 the platform generated $41.8M net revenue from 7,575 distinct bookings made by 1,490 active passengers — an average booking value of $5,519.79. That is roughly $28K of revenue per active passenger over the period, indicating a small but high-value, repeat-booking customer base.

**Management action.** Adopt $5,520 as the benchmark average booking value for next-year targets, and prioritise retention of the 1,490 active passengers since repeat bookings underpin the entire revenue base.

## Q02 — Monthly Revenue Trend Across All Years

Net revenue and bookings by booking month, 1994 through 1998.

In [0]:
%sql
-- Q02 · Monthly net revenue trend across all years
SELECT
  date_format(booking_date, 'yyyy-MM') AS booking_month,
  COUNT(DISTINCT booking_id)            AS bookings,
  ROUND(SUM(net_revenue), 2)            AS net_revenue
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1
ORDER BY 1

**Interpretation.** Monthly net revenue is remarkably stable, oscillating between roughly $592K (Nov 1994) and $858K (Dec 1998) with no recurring seasonal pattern across the 60 months. The strongest months (Dec 1998, Oct 1994, Aug 1998) do not line up as an annual season, suggesting demand is steady business-driven travel rather than holiday peaks.

**Management action.** Plan capacity and pricing on a flat monthly baseline (~$696K average) rather than seasonal assumptions, and investigate whether the late-1998 strength is a trend that can be sustained.

## Q03 — Annual Revenue with Year-over-Year Change

Yearly net revenue with absolute and percentage change versus the prior year.

In [0]:
%sql
-- Q03 · Annual net revenue with year-over-year change
WITH annual AS (
  SELECT
    YEAR(booking_date)         AS booking_year,
    ROUND(SUM(net_revenue), 2) AS net_revenue
  FROM skyledger_airlines.gold.fact_bookings
  GROUP BY 1
)
SELECT
  booking_year,
  net_revenue,
  ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year), 2) AS yoy_change,
  ROUND(100 * (net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year))
        / LAG(net_revenue) OVER (ORDER BY booking_year), 2)             AS yoy_pct_change
FROM annual
ORDER BY booking_year

**Interpretation.** Revenue fell 5.7% in 1995 to $7.95M, then recovered with +4.4% in 1996, +0.9% in 1997 and +4.4% in 1998, finishing at $8.75M — the strongest year of the series. The trajectory is modest single-digit growth with no compounding acceleration.

**Management action.** Set the next year's revenue target at ≥ +4% on 1998, and commission a root-cause review of the 1995 dip to protect against a repeat.

## Q04 — Revenue by Passenger Travel Purpose

Net revenue, bookings and average discount by the passenger's stated travel purpose.

In [0]:
%sql
-- Q04 · Net revenue by passenger travel purpose
SELECT
  p.travel_purpose,
  COUNT(DISTINCT f.booking_id)   AS bookings,
  ROUND(SUM(f.net_revenue), 2)  AS net_revenue,
  ROUND(AVG(f.discount_rate), 4) AS avg_discount_rate
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_passenger p
  ON f.passenger_key = p.passenger_key
GROUP BY 1
ORDER BY net_revenue DESC

**Interpretation.** Revenue is evenly balanced across purposes: BUSINESS leads at $8.95M and CREW trails at $7.93M — a spread of only ~13% across five segments. This diversification cushions revenue against a downturn in any one travel segment.

**Management action.** Preserve the balanced mix while steering incremental growth toward BUSINESS travellers, who combine the highest revenue with the lowest average discount (see Q15).

## Q05 — Top 10 Passengers by Net Revenue

The ten highest-revenue passengers with their travel purpose and home nation.

In [0]:
%sql
-- Q05 · Top 10 passengers by net revenue
SELECT
  p.passenger_name,
  p.travel_purpose,
  p.passenger_nation,
  COUNT(DISTINCT f.booking_id) AS bookings,
  ROUND(SUM(f.net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_passenger p
  ON f.passenger_key = p.passenger_key
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation.** Barbara Petrov is the top customer at $189K net revenue, and every top-10 passenger contributes between $135K and $189K — about $1.55M combined. With only 1,490 active passengers, these customers generate roughly 25× the revenue of an average passenger.

**Management action.** Enrol the top 10 passengers in a premium relationship program (dedicated support, loyalty upgrades) to protect roughly $1.55M of five-year revenue.

## Q06 — Top 10 Passengers as a Share of Total Revenue

Top-10 passenger revenue expressed as a share of the top-10 total and of overall net revenue.

In [0]:
%sql
-- Q06 · Top 10 passengers as a share of total net revenue
WITH top_10_passengers AS (
  SELECT
    p.passenger_name,
    ROUND(SUM(f.net_revenue), 2) AS net_revenue
  FROM skyledger_airlines.gold.fact_bookings f
  JOIN skyledger_airlines.gold.dim_passenger p
    ON f.passenger_key = p.passenger_key
  GROUP BY 1
  ORDER BY 2 DESC
  LIMIT 10
),
total AS (
  SELECT SUM(net_revenue) AS total_net_revenue
  FROM skyledger_airlines.gold.fact_bookings
)
SELECT
  t.passenger_name,
  t.net_revenue,
  ROUND(100 * t.net_revenue / SUM(t.net_revenue) OVER (), 2) AS pct_of_top_10,
  ROUND(100 * t.net_revenue / x.total_net_revenue, 2)         AS pct_of_total_revenue
FROM top_10_passengers t
CROSS JOIN total x
ORDER BY t.net_revenue DESC

**Interpretation.** The top 10 passengers account for only 3.7% of total net revenue — customer-level concentration risk is minimal. Revenue is broadly distributed across the 1,490 active passengers, so no single customer loss is material on its own.

**Management action.** Shift effort from protecting individual whales toward raising average passenger value — replicate the booking frequency of the top decile across the broader base.

## Q07 — Top 10 Routes by Net Revenue

The ten highest-revenue routes with route type and booking counts.

In [0]:
%sql
-- Q07 · Top 10 routes by net revenue
SELECT
  r.route_name,
  r.route_type,
  COUNT(DISTINCT f.booking_id) AS bookings,
  ROUND(SUM(f.net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_route r
  ON f.route_key = r.route_key
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation.** NRT-BOG leads with $923K, followed by LOS-MIA ($888K) and JFK-MAD ($882K); the top routes are predominantly long-haul and ultra-long-haul international pairs. A tight top-10 cluster, each above roughly $800K, forms the revenue backbone of the network.

**Management action.** Guarantee seat availability and monitor competitor entry on these backbone routes — especially NRT-BOG — since a capacity or pricing slip there hits revenue hardest.

## Q08 — Bottom 10 Routes by Net Revenue

The ten lowest-revenue routes with route type and booking counts.

In [0]:
%sql
-- Q08 · Bottom 10 routes by net revenue
SELECT
  r.route_name,
  r.route_type,
  COUNT(DISTINCT f.booking_id) AS bookings,
  ROUND(SUM(f.net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_route r
  ON f.route_key = r.route_key
GROUP BY 1, 2
ORDER BY net_revenue ASC
LIMIT 10

**Interpretation.** The weakest route, AMS-SCL, still earns $348K, and the gap between the best and worst route is only about 2.6× — there are no dead routes in this network. Bottom performers are a mix of short-haul and long-haul pairs plus one domestic route (NBO-SCL).

**Management action.** Before reallocating capacity, benchmark the bottom 10 routes on load factor and operating cost, and test targeted promotions first — even the weakest routes still contribute ~$350K each.

## Q09 — Revenue by Route Type

Net revenue and revenue share split by domestic, short-haul, long-haul and ultra-long-haul route types.

In [0]:
%sql
-- Q09 · Net revenue by route type (domestic / short-haul / long-haul / ultra-long-haul)
SELECT
  r.route_type,
  COUNT(DISTINCT f.booking_id) AS bookings,
  ROUND(SUM(f.net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(f.net_revenue) / SUM(SUM(f.net_revenue)) OVER (), 2) AS revenue_share_pct
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_route r
  ON f.route_key = r.route_key
GROUP BY 1
ORDER BY net_revenue DESC

**Interpretation.** ULTRA_LONG_HAUL routes generate the most revenue ($11.79M, 28.2% of total), followed by LONG_HAUL ($10.92M), SHORT_HAUL ($9.84M) and DOMESTIC ($9.26M). Longer stages monetise better, consistent with higher base fares on intercontinental routes.

**Management action.** Prioritise future capacity growth on ultra-long-haul routes, and reposition domestic routes as feeders that build loyalty rather than direct revenue engines.

## Q10 — Top 10 Airlines by Net Revenue

The ten highest-revenue partner airlines.

In [0]:
%sql
-- Q10 · Top 10 airlines by net revenue
SELECT
  a.airline_name,
  a.airline_region,
  a.performance_rating,
  COUNT(DISTINCT f.booking_id) AS bookings,
  ROUND(SUM(f.net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_airline a
  ON f.airline_key = a.airline_key
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10

**Interpretation.** Eastern Pacific leads at $4.26M with AmeriJet only ~$1.5K behind and Gulf Express at $4.19M — the top carriers are tightly clustered, and no single airline holds more than ~10.2% of platform revenue. Airline geography here is the carrier's home region (supply-side view).

**Management action.** Use the balanced carrier mix as negotiating leverage in partner agreements, and review the lower-ranked airlines for performance-rating or capacity gaps before renewals.

## Q11 — Revenue Share of the Top 10 Airlines

Top-10 airline revenue expressed as a share of total net revenue.

In [0]:
%sql
-- Q11 · Revenue share of the top 10 airlines
WITH top_10_airlines AS (
  SELECT
    a.airline_name,
    ROUND(SUM(f.net_revenue), 2) AS net_revenue
  FROM skyledger_airlines.gold.fact_bookings f
  JOIN skyledger_airlines.gold.dim_airline a
    ON f.airline_key = a.airline_key
  GROUP BY 1
  ORDER BY 2 DESC
  LIMIT 10
),
total AS (
  SELECT SUM(net_revenue) AS total_net_revenue
  FROM skyledger_airlines.gold.fact_bookings
)
SELECT
  t.airline_name,
  t.net_revenue,
  ROUND(100 * t.net_revenue / SUM(t.net_revenue) OVER (), 2) AS pct_of_top_10,
  ROUND(100 * t.net_revenue / x.total_net_revenue, 2)         AS pct_of_total_revenue
FROM top_10_airlines t
CROSS JOIN total x
ORDER BY t.net_revenue DESC

**Interpretation.** The platform is served by exactly 10 airlines, so the top 10 carriers represent 100% of net revenue; even the largest, Eastern Pacific, holds only ~10.2%. Concentration is structurally complete but evenly spread across partners.

**Management action.** Onboard 2–3 additional partner airlines to create supply-side optionality — with 100% of revenue flowing through just 10 carriers, a single partner exit would remove roughly a tenth of revenue.

## Q12 — Revenue by Passenger Region

Net revenue by the passenger's home region — an explicit passenger-geography view.

In [0]:
%sql
-- Q12 · Net revenue by passenger region (passenger geography)
SELECT
  passenger_region AS passenger_home_region,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1
ORDER BY net_revenue DESC

**Interpretation.** By passenger home region, EUROPE generates the most revenue ($9.06M, 21.7%) and AMERICA the least ($7.72M), a spread of about 17% across the five regions. This is a demand-side view — it reflects where passengers live, not where airlines are based (see Q20 for the contrast).

**Management action.** Fund demand-generation campaigns in AMERICA to close the gap with EUROPE, and defend the EUROPE base with loyalty benefits.

## Q13 — Revenue by Passenger Nation (All Nations)

Net revenue for every passenger home nation, with its parent region.

In [0]:
%sql
-- Q13 · Net revenue by passenger nation (all nations)
SELECT
  passenger_nation,
  passenger_region,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1, 2
ORDER BY net_revenue DESC

**Interpretation.** France ($2.24M), Turkey ($1.98M) and Kenya ($1.88M) head a list of 25 passenger nations, with revenue decaying gradually rather than collapsing — the customer base is genuinely global and no single nation exceeds ~5.4% of total revenue.

**Management action.** Localise marketing, payment methods and fare products for the top five nations, and watch the long tail for emerging markets worth dedicated route investment.

## Q14 — Revenue and Volume Split by Refund Flag

Net revenue, distinct bookings and segment volume split by the refund flag recorded on each flight segment.

In [0]:
%sql
-- Q14 · Revenue and volume split by refund flag
-- refund_flag is recorded per flight segment ('N', 'A', 'R'); a booking can span several flags
SELECT
  refund_flag,
  COUNT(DISTINCT booking_id) AS bookings,
  COUNT(*)                   AS segment_lines,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1
ORDER BY net_revenue DESC

**Interpretation.** Segments flagged 'N' carry $27.27M — 65.2% of revenue across 6,841 distinct bookings — while 'A' holds $10.42M and 'R' $4.13M (2,132 bookings), so refund-related activity touches roughly a third of net revenue. Because the flag lives on each flight segment, one booking can combine multiple flags, so booking counts across flags overlap.

**Management action.** Size the margin cost of refund-eligible fares and pilot tighter refund rules on low-value segments while preserving flexibility for high-value customers.

## Q15 — Average Discount by Travel Purpose

Mean discount rate applied to flight segments, by passenger travel purpose.

In [0]:
%sql
-- Q15 · Average discount by travel purpose
SELECT
  p.travel_purpose,
  COUNT(*)                            AS segment_lines,
  ROUND(AVG(f.discount_rate), 4)     AS avg_discount_rate
FROM skyledger_airlines.gold.fact_bookings f
JOIN skyledger_airlines.gold.dim_passenger p
  ON f.passenger_key = p.passenger_key
GROUP BY 1
ORDER BY avg_discount_rate DESC

**Interpretation.** Average discount is almost flat across purposes — LEISURE is highest at ~7.0% and BUSINESS lowest at ~6.8%. Discounting is effectively undifferentiated by customer segment, which suggests pricing policy has not been tuned to willingness to pay.

**Management action.** Introduce segment-level discount controls and A/B test lower discounts on less price-sensitive segments (e.g., BUSINESS, MILITARY) to recover margin.

## Q16 — Average Flight Duration by Cabin Class

Average trip duration in days from booking departure date to arrival date, by cabin class.

In [0]:
%sql
-- Q16 · Average flight duration (departure → arrival, in days) by cabin class
SELECT
  cabin_class,
  COUNT(*)                                                    AS segment_lines,
  ROUND(AVG(DATEDIFF(arrival_date, departure_date)), 2)       AS avg_trip_days
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1
ORDER BY avg_trip_days DESC, cabin_class

**Interpretation.** Average trip duration is 0.0 days in every cabin class — all 24,666 segments record the arrival date on the same calendar day as departure. This is a limitation of the TPC-A practice data (same-day arrivals), so duration cannot differentiate cabin demand in this dataset.

**Management action.** Do not use duration-based cabin insights until true arrival timestamps are captured; raise a data-sourcing request for real departure/arrival times in the next data-quality cycle.

## Q17 — Booking Count and Revenue by Booking Priority

Distinct bookings, net revenue and average booking value across the five booking-priority tiers.

In [0]:
%sql
-- Q17 · Booking count and revenue by booking priority
SELECT
  booking_priority,
  COUNT(DISTINCT booking_id)                              AS bookings,
  ROUND(SUM(net_revenue), 2)                              AS net_revenue,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2)  AS average_booking_value
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1
ORDER BY 1

**Interpretation.** Bookings and revenue are almost perfectly uniform across the five priorities — each tier earns $8.0–8.7M from roughly 1,470–1,600 bookings. Average booking value is also flat (5-STANDARD nominally highest at $5,601), so the priority flag carries no revenue signal in this dataset.

**Management action.** Audit whether booking priority reflects a real operational process; if urgent bookings are meant to command a premium, revisit the pricing rules that currently ignore urgency.

## Q18 — Monthly Breakdown of Booking Status

Distinct bookings and net revenue by month and booking status ('F', 'O', 'C'), 1994–1998.

In [0]:
%sql
-- Q18 · Monthly breakdown of booking status
SELECT
  date_format(booking_date, 'yyyy-MM') AS booking_month,
  booking_status,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM skyledger_airlines.gold.fact_bookings
GROUP BY 1, 2
ORDER BY 1,
  CASE booking_status WHEN 'F' THEN 1 WHEN 'O' THEN 2 ELSE 3 END

**Interpretation.** Finalised ('F') bookings dominate every month — $31.35M / ~75% of revenue and 5,696 bookings overall — versus open ('O', $6.17M) and cancelled ('C', $4.30M). The monthly status mix stays stable across all 60 months with no cancellation spikes.

**Management action.** Use the stable historical F-share as the baseline for revenue forecasting, and configure an alert if any month's cancelled bookings or revenue share breaks materially from the historical pattern.

## Q19 — Route Revenue Ranking Within Each Travel Purpose

Revenue ranking of routes within each travel purpose (top 5 routes per purpose).

In [0]:
%sql
-- Q19 · Route revenue ranking within each travel purpose (top 5 per purpose)
WITH route_purpose_revenue AS (
  SELECT
    p.travel_purpose,
    r.route_name,
    ROUND(SUM(f.net_revenue), 2) AS net_revenue
  FROM skyledger_airlines.gold.fact_bookings f
  JOIN skyledger_airlines.gold.dim_passenger p
    ON f.passenger_key = p.passenger_key
  JOIN skyledger_airlines.gold.dim_route r
    ON f.route_key = r.route_key
  GROUP BY 1, 2
),
ranked AS (
  SELECT
    travel_purpose,
    route_name,
    net_revenue,
    ROW_NUMBER() OVER (PARTITION BY travel_purpose ORDER BY net_revenue DESC) AS revenue_rank
  FROM route_purpose_revenue
)
SELECT
  travel_purpose,
  revenue_rank,
  route_name,
  net_revenue
FROM ranked
WHERE revenue_rank <= 5
ORDER BY travel_purpose, revenue_rank

**Interpretation.** Each travel purpose has a distinct route profile: BUSINESS and MILITARY peak on NRT-BOG, CREW on LAX-CPT, LEISURE on BOM-IST, and STUDENT on LOS-MIA. The number-one route per purpose earns roughly $180K–$210K, so the differences are meaningful but not dramatic.

**Management action.** Tailor route-level offers by purpose — student fares on LOS-MIA, business-flex products on NRT-BOG — rather than running one-size-fits-all route promotions.

## Q20 — Passenger Geography vs Airline Geography by Region

Net revenue by region computed two ways — where the passenger lives versus where the carrier is headquartered — side by side.

In [0]:
%sql
-- Q20 · Revenue by region: passenger geography vs airline geography
WITH passenger_geography AS (
  SELECT
    passenger_region AS region_name,
    ROUND(SUM(net_revenue), 2) AS passenger_geography_revenue
  FROM skyledger_airlines.gold.fact_bookings
  GROUP BY 1
),
airline_geography AS (
  SELECT
    airline_region AS region_name,
    ROUND(SUM(net_revenue), 2) AS airline_geography_revenue
  FROM skyledger_airlines.gold.fact_bookings
  GROUP BY 1
)
SELECT
  COALESCE(p.region_name, a.region_name) AS region_name,
  p.passenger_geography_revenue,
  a.airline_geography_revenue
FROM passenger_geography p
FULL OUTER JOIN airline_geography a
  ON p.region_name = a.region_name
ORDER BY region_name

**Interpretation.** Revenue by passenger home region ranges from $7.72M (AMERICA) to $9.06M (EUROPE), while revenue by airline home region is nearly flat ($8.31M–$8.44M across the same five regions). The variation is almost entirely demand-side: where passengers live matters far more than where carriers are headquartered.

**Management action.** Allocate growth investment by passenger geography rather than airline nationality, and use the airline-geography view only to monitor supply-side partner balance.